# GPT-2 Small vs Medium 指令微调模型评测

这个 notebook 用于对比本地两个 SFT checkpoint 的表现：

- `gpt2-small124M-sft.pth`
- `gpt2-medium355M-sft.pth`

评测内容：

1. 使用和训练时一致的 Alpaca 风格 prompt 生成测试集回答
2. 抽样并排查看标准答案、small 回答、medium 回答
3. 使用智谱 AI `glm-4-flash` 作为 LLM-as-Judge，对每条回答进行 0-100 分评分并给出简短理由
4. 汇总两个模型的平均分、胜负关系和典型样本
5. 导出 JSON/CSV，方便后续人工检查

运行智谱评测前，请先设置环境变量 `ZHIPU_API_KEY`。

In [ ]:
from pathlib import Path
import gc
import json
import math
import os
import random
import re
import time

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import tiktoken
from tqdm.auto import tqdm


def find_project_root():
    """支持从项目根目录或 ch07 目录启动 notebook。"""
    for path in [Path.cwd(), *Path.cwd().parents]:
        if (path / "instruction-data.json").exists():
            return path
    raise FileNotFoundError("未找到 instruction-data.json，请确认 notebook 在项目目录内运行。")


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "instruction-data.json"
OUTPUT_JSON = PROJECT_ROOT / "sft-model-comparison-responses.json"
OUTPUT_CSV = PROJECT_ROOT / "sft-model-comparison-responses.csv"
JUDGE_OUTPUT_JSON = PROJECT_ROOT / "sft-model-comparison-zhipu-judge.json"
JUDGE_OUTPUT_CSV = PROJECT_ROOT / "sft-model-comparison-zhipu-judge.csv"

MODEL_SPECS = {
    "small124M": {
        "checkpoint": PROJECT_ROOT / "gpt2-small124M-sft.pth",
        "config": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    },
    "medium355M": {
        "checkpoint": PROJECT_ROOT / "gpt2-medium355M-sft.pth",
        "config": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    },
}

BASE_CONFIG = {
    "vocab_size": 50257,
    "context_length": 1024,
    "drop_rate": 0.0,
    "qkv_bias": True,
}

# None 表示评测完整测试集；调试时可以改成 10 或 30。
EVAL_LIMIT = None
MAX_NEW_TOKENS = 256
USE_CACHE = True
RANDOM_SEED = 42

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

tokenizer = tiktoken.get_encoding("gpt2")

print(f"项目根目录: {PROJECT_ROOT}")
print(f"使用设备: {device}")
for name, spec in MODEL_SPECS.items():
    print(f"{name}: {spec['checkpoint']} | exists={spec['checkpoint'].exists()}")

In [ ]:
with DATA_PATH.open("r", encoding="utf-8") as f:
    data = json.load(f)

train_portion = int(len(data) * 0.85)
test_portion = int(len(data) * 0.1)

train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
val_data = data[train_portion + test_portion:]

eval_data = test_data if EVAL_LIMIT is None else test_data[:EVAL_LIMIT]

print(f"总数据: {len(data)} 条")
print(f"训练集: {len(train_data)} | 测试集: {len(test_data)} | 验证集: {len(val_data)}")
print(f"本次评测: {len(eval_data)} 条")
print(f"字段: {list(eval_data[0].keys())}")

## 模型结构

下面复用第 4-7 章的 GPTModel 结构。这里不再加载 Hugging Face 原始权重，而是直接加载已经微调好的 `.pth` checkpoint。

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        queries = self.W_query(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        keys = self.W_key(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values = self.W_value(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        attn_weights = torch.softmax(attn_scores / math.sqrt(self.head_dim), dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vecs = (attn_weights @ values).transpose(1, 2)
        context_vecs = context_vecs.contiguous().view(b, num_tokens, self.d_out)
        return self.out_proj(context_vecs)


class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        return self.scale * (x - mean) / torch.sqrt(var + self.eps) + self.shift


class GELU(nn.Module):
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(math.sqrt(2.0 / math.pi) * (x + 0.044715 * torch.pow(x, 3))))


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            dropout=cfg["drop_rate"],
            num_heads=cfg["n_heads"],
            qkv_bias=cfg["qkv_bias"],
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        x = x + self.drop_shortcut(self.att(self.norm1(x)))
        x = x + self.drop_shortcut(self.ff(self.norm2(x)))
        return x


class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        return self.out_head(x)


print("GPTModel 定义完成")

In [ ]:
def format_input(entry):
    instruction_text = (
        "Below is an instruction that describes a task. "
        "Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )
    input_text = f"\n\n### Input:\n{entry['input']}" if entry.get("input") else ""
    return instruction_text + input_text


def format_prompt(entry):
    return format_input(entry) + "\n\n### Response:\n"


def text_to_token_ids(text, tokenizer):
    return torch.tensor(tokenizer.encode(text, allowed_special={"<|endoftext|>"})).unsqueeze(0)


def token_ids_to_text(token_ids, tokenizer):
    return tokenizer.decode(token_ids.squeeze(0).tolist())


def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]

        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(logits < min_val, torch.tensor(float("-inf"), device=logits.device), logits)

        if temperature > 0.0:
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)

        if eos_id is not None and idx_next.item() == eos_id:
            break
        idx = torch.cat((idx, idx_next), dim=1)
    return idx


def cleanup_response(response):
    response = response.replace("<|endoftext|>", "").strip()
    response = re.sub(r"^### Response:\s*", "", response).strip()
    # 如果模型继续生成下一个样本模板，只保留当前回答。
    for marker in ["\n\n### Instruction:", "\n\n### Input:", "\n\nBelow is an instruction"]:
        if marker in response:
            response = response.split(marker, 1)[0].strip()
    return response


def build_model_config(model_name):
    cfg = BASE_CONFIG.copy()
    cfg.update(MODEL_SPECS[model_name]["config"])
    return cfg


def load_sft_model(model_name, device):
    spec = MODEL_SPECS[model_name]
    checkpoint_path = spec["checkpoint"]
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"找不到 checkpoint: {checkpoint_path}")

    cfg = build_model_config(model_name)
    model = GPTModel(cfg)
    state_dict = torch.load(checkpoint_path, map_location="cpu")
    model.load_state_dict(state_dict, strict=True)
    model.to(device)
    model.eval()

    n_params = sum(p.numel() for p in model.parameters())
    print(f"{model_name} 加载完成 | 参数量: {n_params:,}")
    return model, cfg


def free_model(model):
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if hasattr(torch, "mps") and torch.backends.mps.is_available():
        torch.mps.empty_cache()


print("生成与加载函数定义完成")

## 生成两个模型的测试集回答

默认会评测完整测试集，并把结果缓存到 `sft-model-comparison-responses.json`。如果你修改了 `EVAL_LIMIT`、`MAX_NEW_TOKENS` 或重新训练了模型，可以把 `USE_CACHE` 改成 `False` 后重新运行。

In [ ]:
def cache_matches(cache, eval_data):
    if not isinstance(cache, list) or len(cache) != len(eval_data):
        return False
    required_keys = {"instruction", "input", "output", *MODEL_SPECS.keys()}
    return all(required_keys.issubset(item.keys()) for item in cache)


def generate_responses_for_model(model_name, eval_data):
    model, cfg = load_sft_model(model_name, device)
    responses = []
    start_time = time.time()

    for entry in tqdm(eval_data, desc=f"生成 {model_name} 回答"):
        prompt = format_prompt(entry)
        input_ids = text_to_token_ids(prompt, tokenizer).to(device)
        token_ids = generate(
            model=model,
            idx=input_ids,
            max_new_tokens=MAX_NEW_TOKENS,
            context_size=cfg["context_length"],
            eos_id=50256,
        )
        full_text = token_ids_to_text(token_ids.cpu(), tokenizer)
        response = cleanup_response(full_text[len(prompt):])
        responses.append(response)

    elapsed = time.time() - start_time
    print(f"{model_name} 完成 | {len(eval_data)} 条 | {elapsed / 60:.2f} 分钟")
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if hasattr(torch, "mps") and torch.backends.mps.is_available():
        torch.mps.empty_cache()
    return responses


if USE_CACHE and OUTPUT_JSON.exists():
    with OUTPUT_JSON.open("r", encoding="utf-8") as f:
        comparison_results = json.load(f)
    if cache_matches(comparison_results, eval_data):
        print(f"已从缓存读取: {OUTPUT_JSON}")
    else:
        print("缓存内容和当前评测配置不匹配，将重新生成。")
        comparison_results = None
else:
    comparison_results = None

if comparison_results is None:
    comparison_results = [
        {
            "index": i,
            "instruction": entry["instruction"],
            "input": entry.get("input", ""),
            "output": entry["output"],
        }
        for i, entry in enumerate(eval_data)
    ]

    for model_name in MODEL_SPECS:
        model_responses = generate_responses_for_model(model_name, eval_data)
        for item, response in zip(comparison_results, model_responses):
            item[model_name] = response

    with OUTPUT_JSON.open("w", encoding="utf-8") as f:
        json.dump(comparison_results, f, ensure_ascii=False, indent=2)
    print(f"结果已保存: {OUTPUT_JSON}")

print(f"结果条数: {len(comparison_results)}")

## 抽样并排查看

这一步用于人工快速感受两个模型回答差异。可以调整 `N_SAMPLES` 或 `RANDOM_SEED`。

In [ ]:
N_SAMPLES = min(10, len(comparison_results))
random.seed(RANDOM_SEED)
samples = random.sample(comparison_results, N_SAMPLES)

for n, item in enumerate(samples, start=1):
    print(f"{'=' * 90}")
    print(f"样本 {n} | 原测试集索引: {item['index']}")
    print(f"指令: {item['instruction']}")
    if item["input"]:
        print(f"输入: {item['input']}")
    print(f"标准答案: {item['output']}")
    print("-" * 90)
    print(f"small124M : {item['small124M']}")
    print(f"medium355M: {item['medium355M']}")
    print()

## 智谱 AI LLM-as-Judge 评测

下面使用智谱 AI `glm-4-flash` 对每个模型回答进行评分。评估模型会同时看到指令、可选输入、标准答案和模型回答，并返回 JSON：`score` 为 0-100 分，`reason` 为简短理由。

注意：这里不再使用完全匹配、关键词覆盖、回答长度等简单自动指标。

In [ ]:
try:
    from zhipuai import ZhipuAI
except ImportError as exc:
    raise ImportError("请先安装 zhipuai：pip install zhipuai") from exc

ZHIPU_API_KEY = 'd63eddac939145a1a2564bfb51a58ac9.OM0UPDTi3lBFNMcu'
EVAL_MODEL = "glm-4-flash"
USE_JUDGE_CACHE = True

if not ZHIPU_API_KEY:
    raise ValueError("未找到环境变量 ZHIPU_API_KEY。请先设置后再运行智谱评测。")

client = ZhipuAI(api_key=ZHIPU_API_KEY)

test_resp = client.chat.completions.create(
    model=EVAL_MODEL,
    messages=[{"role": "user", "content": "回复数字 42"}],
    temperature=0.0,
    max_tokens=20,
)
print(f"智谱 AI 连接测试: {test_resp.choices[0].message.content.strip()} ✓")

In [ ]:
def build_judge_prompt(item, model_name):
    input_part = item["input"] if item["input"] else "(none)"
    return f"""
You are an impartial evaluator for instruction-following language models.

Evaluate the model response against the user instruction, optional input, and reference answer.
Score from 0 to 100:
- 100: fully correct, follows the instruction, equivalent to or better than the reference.
- 80: mostly correct with minor wording or completeness issues.
- 60: partially correct but misses important details.
- 40: related but mostly wrong or incomplete.
- 20: barely related.
- 0: empty, irrelevant, or contradicts the task.

Return JSON only, with exactly these keys:
{{"score": <integer 0-100>, "reason": "<brief reason in Chinese>"}}

Instruction:
{item['instruction']}

Input:
{input_part}

Reference answer:
{item['output']}

Model name:
{model_name}

Model response:
{item[model_name]}
""".strip()


def parse_judge_response(raw_text):
    raw_text = raw_text.strip()
    try:
        parsed = json.loads(raw_text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", raw_text, flags=re.S)
        if not match:
            raise
        parsed = json.loads(match.group(0))

    score = int(parsed["score"])
    score = max(0, min(100, score))
    reason = str(parsed.get("reason", "")).strip()
    return score, reason


def judge_one_response(item, model_name):
    prompt = build_judge_prompt(item, model_name)
    response = client.chat.completions.create(
        model=EVAL_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=160,
    )
    raw_text = response.choices[0].message.content.strip()
    score, reason = parse_judge_response(raw_text)
    return {"score": score, "reason": reason, "raw_judge_response": raw_text}


print("智谱评测函数定义完成")

## 运行智谱评分

每条测试数据会分别评估 `small124M` 和 `medium355M` 两个回答。结果会缓存到 `sft-model-comparison-zhipu-judge.json`，避免重复调用 API。

In [ ]:
def judge_cache_matches(cache, comparison_results):
    if not isinstance(cache, list) or len(cache) != len(comparison_results):
        return False
    for item in cache:
        if "judgements" not in item:
            return False
        if not all(model_name in item["judgements"] for model_name in MODEL_SPECS):
            return False
    return True


if USE_JUDGE_CACHE and JUDGE_OUTPUT_JSON.exists():
    with JUDGE_OUTPUT_JSON.open("r", encoding="utf-8") as f:
        judged_results = json.load(f)
    if judge_cache_matches(judged_results, comparison_results):
        print(f"已从缓存读取智谱评分: {JUDGE_OUTPUT_JSON}")
    else:
        print("智谱评分缓存不匹配，将重新评测。")
        judged_results = None
else:
    judged_results = None

if judged_results is None:
    judged_results = []
    for item in tqdm(comparison_results, desc="智谱 AI 评分"):
        judged_item = dict(item)
        judged_item["judgements"] = {}
        for model_name in MODEL_SPECS:
            try:
                judged_item["judgements"][model_name] = judge_one_response(item, model_name)
            except Exception as exc:
                judged_item["judgements"][model_name] = {
                    "score": None,
                    "reason": f"评分失败: {exc}",
                    "raw_judge_response": "",
                }
        judged_results.append(judged_item)

    with JUDGE_OUTPUT_JSON.open("w", encoding="utf-8") as f:
        json.dump(judged_results, f, ensure_ascii=False, indent=2)
    print(f"智谱评分已保存: {JUDGE_OUTPUT_JSON}")

print(f"智谱评分条数: {len(judged_results)}")

In [ ]:
judge_rows = []
for item in judged_results:
    row = {
        "index": item["index"],
        "instruction": item["instruction"],
        "input": item["input"],
        "output": item["output"],
        "small124M": item["small124M"],
        "medium355M": item["medium355M"],
    }
    for model_name in MODEL_SPECS:
        judgement = item["judgements"][model_name]
        row[f"{model_name}_score"] = judgement["score"]
        row[f"{model_name}_reason"] = judgement["reason"]
    row["medium_minus_small"] = (
        row["medium355M_score"] - row["small124M_score"]
        if row["medium355M_score"] is not None and row["small124M_score"] is not None
        else None
    )
    if row["medium_minus_small"] is None:
        row["winner"] = "评分失败"
    elif row["medium_minus_small"] > 0:
        row["winner"] = "medium355M"
    elif row["medium_minus_small"] < 0:
        row["winner"] = "small124M"
    else:
        row["winner"] = "tie"
    judge_rows.append(row)

judge_df = pd.DataFrame(judge_rows)
summary_df = pd.DataFrame(
    [
        {
            "model": model_name,
            "valid_scores": judge_df[f"{model_name}_score"].notna().sum(),
            "mean_score": judge_df[f"{model_name}_score"].mean(),
            "median_score": judge_df[f"{model_name}_score"].median(),
            "min_score": judge_df[f"{model_name}_score"].min(),
            "max_score": judge_df[f"{model_name}_score"].max(),
        }
        for model_name in MODEL_SPECS
    ]
)

print("胜负统计:")
print(judge_df["winner"].value_counts(dropna=False))
summary_df

## 查看智谱评分结果

下面只基于智谱 AI 的评分做可视化和样本排序：平均分越高表示整体回答质量越接近标准答案，`medium_minus_small` 越大表示 medium 在该样本上越占优。

In [ ]:
plot_df = judge_df.dropna(subset=["small124M_score", "medium355M_score", "medium_minus_small"]).copy()

colors = {
    "small124M": "#4C78A8",
    "medium355M": "#F58518",
    "tie": "#9CA3AF",
}

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Zhipu AI Judge: GPT-2 Small vs Medium SFT", fontsize=16, fontweight="bold")

# 1) 平均分 + 中位数
score_summary = summary_df.set_index("model")[["mean_score", "median_score"]]
score_summary.plot(
    kind="bar",
    ax=axes[0, 0],
    color=["#6BAED6", "#FD8D3C"],
    ylim=(0, 100),
    rot=0,
)
axes[0, 0].set_title("Mean / Median Score")
axes[0, 0].set_ylabel("score")
axes[0, 0].bar_label(axes[0, 0].containers[0], fmt="%.1f", padding=3)
axes[0, 0].bar_label(axes[0, 0].containers[1], fmt="%.1f", padding=3)

# 2) 分数分布
bins = list(range(0, 101, 10))
axes[0, 1].hist(
    plot_df["small124M_score"],
    bins=bins,
    alpha=0.65,
    label="small124M",
    color=colors["small124M"],
)
axes[0, 1].hist(
    plot_df["medium355M_score"],
    bins=bins,
    alpha=0.65,
    label="medium355M",
    color=colors["medium355M"],
)
axes[0, 1].set_title("Score Distribution")
axes[0, 1].set_xlabel("score")
axes[0, 1].set_ylabel("count")
axes[0, 1].legend()

# 3) 每条样本的 small vs medium 散点对比
axes[0, 2].scatter(
    plot_df["small124M_score"],
    plot_df["medium355M_score"],
    c=plot_df["medium_minus_small"].map(lambda x: colors["medium355M"] if x > 0 else colors["small124M"] if x < 0 else colors["tie"]),
    alpha=0.75,
    edgecolors="white",
    linewidths=0.6,
)
axes[0, 2].plot([0, 100], [0, 100], linestyle="--", color="#6B7280", linewidth=1)
axes[0, 2].set_xlim(0, 100)
axes[0, 2].set_ylim(0, 100)
axes[0, 2].set_title("Per-sample Score: Small vs Medium")
axes[0, 2].set_xlabel("small124M score")
axes[0, 2].set_ylabel("medium355M score")
axes[0, 2].text(5, 92, "above line: medium wins", color=colors["medium355M"])
axes[0, 2].text(50, 8, "below line: small wins", color=colors["small124M"])

# 4) 胜负数量
winner_counts = plot_df["winner"].value_counts().reindex(["medium355M", "small124M", "tie"], fill_value=0)
winner_counts.plot(
    kind="bar",
    ax=axes[1, 0],
    color=[colors["medium355M"], colors["small124M"], colors["tie"]],
    rot=0,
)
axes[1, 0].set_title("Win Count by Zhipu Score")
axes[1, 0].set_ylabel("samples")
axes[1, 0].bar_label(axes[1, 0].containers[0], padding=3)

# 5) 分差分布：正数表示 medium 更好
axes[1, 1].hist(plot_df["medium_minus_small"], bins=20, color="#7C3AED", alpha=0.75)
axes[1, 1].axvline(0, color="#111827", linestyle="--", linewidth=1)
axes[1, 1].set_title("Score Difference Distribution")
axes[1, 1].set_xlabel("medium355M score - small124M score")
axes[1, 1].set_ylabel("count")

# 6) 分差绝对值最大的样本
largest_gap = plot_df.reindex(plot_df["medium_minus_small"].abs().sort_values(ascending=False).index).head(10)
bar_colors = [colors["medium355M"] if x > 0 else colors["small124M"] for x in largest_gap["medium_minus_small"]]
axes[1, 2].barh(largest_gap["index"].astype(str), largest_gap["medium_minus_small"], color=bar_colors)
axes[1, 2].axvline(0, color="#111827", linestyle="--", linewidth=1)
axes[1, 2].invert_yaxis()
axes[1, 2].set_title("Top 10 Largest Score Gaps")
axes[1, 2].set_xlabel("medium355M - small124M")
axes[1, 2].set_ylabel("test index")

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

print(f"平均分差 medium-small: {plot_df['medium_minus_small'].mean():.2f}")
print(f"medium 胜出: {(plot_df['medium_minus_small'] > 0).sum()} 条")
print(f"small 胜出 : {(plot_df['medium_minus_small'] < 0).sum()} 条")
print(f"平局       : {(plot_df['medium_minus_small'] == 0).sum()} 条")

In [ ]:
judge_df.sort_values("medium_minus_small", ascending=True).head(10)

## 导出智谱评测结果

CSV 适合用表格软件快速浏览；JSON 保留完整结构，包括每条回答的智谱评分原始返回。

In [ ]:
judge_df.to_csv(JUDGE_OUTPUT_CSV, index=False)

with JUDGE_OUTPUT_JSON.open("w", encoding="utf-8") as f:
    json.dump(judged_results, f, ensure_ascii=False, indent=2)

summary_path = PROJECT_ROOT / "sft-model-comparison-zhipu-summary.csv"
summary_df.to_csv(summary_path, index=False)

print(f"智谱评分 JSON: {JUDGE_OUTPUT_JSON}")
print(f"智谱评分 CSV : {JUDGE_OUTPUT_CSV}")
print(f"汇总 CSV     : {summary_path}")